# ROI Mask Caching

Extends the existing full-mammogram / cropped-lesion caching (`cbis-ddsm-cache-indexed/{split}/{full,crop}/`,
produced by the (now-lost) original `png_caching` run behind `dataframes/{train,val,test}_df_final.csv`) with
a third modality: the ROI (Region of Interest) segmentation mask.

**Key fact used here:** in CBIS-DDSM, the cropped-lesion image and its ROI mask are two separate DICOM
instances stored in the *same* series-UID folder (confirmed by inspecting `data_cleaning_three_input.ipynb`'s
raw metadata columns — `cropped image file path` and `ROI mask file path` resolve to an identical folder for
a given abnormality). After DICOM→PNG conversion that folder holds exactly two PNGs. `train_df_final.csv`
already tells us, per row, exactly which of those two PNGs is the crop (`cropped image file path`) — so the
ROI mask is simply **whichever other PNG is in that same folder**. No path-matching against the separate
`three_input_*.csv` dataframes, and no pixel-variance/size heuristics, are needed.

Verified on a random sample of 60 rows: 56/60 folders contained exactly 2 PNGs (crop + mask), and the file
matching the known crop path was always the smaller-resolution one — the mask is stored at full-mammogram
resolution (thousands of pixels), confirming it's a whole-image segmentation mask, not a lesion-sized crop.
The remaining ~4/60 folders had only 1 PNG (no mask available for that abnormality) — those rows are dropped.

**Output:** `roi_{idx}.png` cached at 224x224 (matching `full_{idx}.png` / `crop_{idx}.png`) under
`cbis-ddsm-cache-indexed/{split}/roi/`, and `dataframes/{split}_df_with_roi.csv` — a copy of
`{split}_df_final.csv` with an added `roi mask cached path` column, rows without a resolvable mask dropped.
The original `{split}_df_final.csv` files are left untouched so nothing that already depends on them breaks.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from PIL import Image

ROOT = Path(".").resolve().parent  # this notebook lives in notebooks/, project root is one level up
PNG_ROOT = ROOT / "cbis_ddsm_png"
CACHE_ROOT = ROOT / "cbis-ddsm-cache-indexed"
CACHE_SIZE = (224, 224)
SPLITS = ["train", "val", "test"]


def winlong(p):
    """Windows MAX_PATH (260 char) workaround -- CBIS-DDSM's nested DICOM UID
    folders routinely exceed it. Built via chr(92) to sidestep backslash-escaping
    ambiguity rather than hand-counting backslash literals."""
    backslash = chr(92)
    unc_prefix = backslash * 2 + "?" + backslash
    p = str(Path(p).resolve())
    if os.name == "nt" and not p.startswith(unc_prefix):
        p = unc_prefix + p
    return p


In [ ]:
def resolve_roi_mask(crop_rel_path):
    """Given the already-resolved 'cropped image file path' (relative, filename
    included), returns the sibling PNG in the same source folder that is NOT the
    crop itself -- the ROI mask. Returns None if that folder doesn't contain
    exactly one other PNG (ambiguous, or genuinely no mask available)."""
    folder = PNG_ROOT / Path(crop_rel_path).parent
    if not Path(winlong(folder)).exists():
        return None
    pngs = list(Path(winlong(folder)).glob("*.png"))
    crop_basename = Path(crop_rel_path).name
    candidates = [p for p in pngs if p.name != crop_basename]
    if len(pngs) == 2 and len(candidates) == 1:
        return candidates[0]
    return None


In [ ]:
total_summary = {}

for split in SPLITS:
    df = pd.read_csv(ROOT / "dataframes" / f"{split}_df_final.csv")
    roi_dir = CACHE_ROOT / split / "roi"
    roi_dir.mkdir(parents=True, exist_ok=True)

    roi_cached_paths = []
    n_cached, n_reused, n_missing = 0, 0, 0

    for idx, row in df.iterrows():
        mask_path = resolve_roi_mask(row["cropped image file path"])
        if mask_path is None:
            roi_cached_paths.append(None)
            n_missing += 1
            continue

        out_path = roi_dir / f"roi_{idx}.png"
        if out_path.exists():
            n_reused += 1
        else:
            img = Image.open(winlong(mask_path)).convert("L")
            img = img.resize(CACHE_SIZE, Image.NEAREST)  # NEAREST: preserves sharp mask edges (no gray blur)
            img = img.convert("RGB")  # replicate to 3ch so it matches RGB-pretrained backbone input
            img.save(out_path)
            n_cached += 1

        roi_cached_paths.append(str(out_path.resolve()))

    df["roi mask cached path"] = roi_cached_paths
    kept_df = df.dropna(subset=["roi mask cached path"]).reset_index(drop=True)

    out_csv = ROOT / "dataframes" / f"{split}_df_with_roi.csv"
    kept_df.to_csv(out_csv, index=False)

    total_summary[split] = {
        "total_rows": len(df), "cached_new": n_cached, "reused_existing": n_reused,
        "missing_roi": n_missing, "kept_rows": len(kept_df), "out_csv": str(out_csv),
    }
    print(f"[{split}] total={len(df)} cached_new={n_cached} reused={n_reused} "
          f"missing_roi={n_missing} kept={len(kept_df)} -> {out_csv}")

print("\nDone.")
total_summary


## Sanity check

Spot-check a few cached masks: should be 224x224 RGB, near-binary (mask vs. background), with a small
nonzero fraction (the lesion is a small region of the full mammogram).

In [ ]:
import numpy as np

check_dir = CACHE_ROOT / "train" / "roi"
for f in sorted(check_dir.glob("*.png"))[:5]:
    img = Image.open(f)
    arr = np.array(img.convert("L"))
    print(f.name, img.size, img.mode, "| unique values:", len(np.unique(arr)),
          "| nonzero fraction:", round((arr > 10).mean(), 3))
